# MedSAMv2 prolematic cases
This notebook aims at proving that problematic cases in MedSAMv2 segmentation are not related to geometry transformations mismatch but rather wrongfull segmentations by the model

In [4]:
# Handle Mask
from pathlib import Path
import nibabel as nib
import nibabel.orientations as nio
import numpy as np
import pandas as pd
import SimpleITK as sitk

df = pd.read_csv("luna25-annotations.csv")
uid = "1.3.6.1.4.1.14519.5.2.1.7009.9004.305851184008713562567881984816"
annotations_group = df[df["SeriesInstanceUID"] == uid.replace("_", ".")]

mask_dir = Path("/opt/data/beegfs/nn-datasets/Task023_LUNA25/raw_splitted/labelsTr/")
image_dir = Path("/opt/data/slow/LUNA25/luna25_images/")

sanitized_uid = uid.replace(".", "_")
src_mask = mask_dir / f"{sanitized_uid}.nii.gz"
mha_path = image_dir / f"{uid}.mha"

# Load mask (NIfTI)
nii = nib.load(src_mask)
data = nii.get_fdata()
affine = nii.affine
inv_affine = np.linalg.inv(affine)

# Load reference image (MHA)
mha_ref = sitk.ReadImage(str(mha_path))

# --- Geometry diagnostics (helps spot LPS/RAS and axis-order issues) ---
print("Mask NIfTI shape:", data.shape)
print("MHA size (x,y,z):", mha_ref.GetSize())
print("MHA origin (LPS):", mha_ref.GetOrigin())
print("MHA spacing:", mha_ref.GetSpacing())
print("MHA direction:", mha_ref.GetDirection())
print("NIfTI affine origin (RAS):", tuple(float(v) for v in affine[:3, 3]))
nifti_spacing = tuple(float(np.linalg.norm(affine[:3, i])) for i in range(3))
print("NIfTI spacing (from affine):", nifti_spacing)
try:
    axcodes = nio.aff2axcodes(affine)
    print("NIfTI aff2axcodes:", axcodes)
except Exception as e:
    print("Could not compute NIfTI axcodes:", repr(e))
print("Num annotations rows:", len(annotations_group))

# Helpers
def _round_tuple(vals):
    return tuple(int(np.round(v)) for v in vals)

def _floor_tuple(vals):
    return tuple(int(np.floor(v)) for v in vals)

def _ceil_tuple(vals):
    return tuple(int(np.ceil(v)) for v in vals)

def _in_bounds(idx, shape):
    return all(0 <= idx[d] < shape[d] for d in range(3))

def _sample_mask(ix, iy, iz, *, swap_xy=False):
    if swap_xy:
        ix, iy = iy, ix
    if not _in_bounds((ix, iy, iz), data.shape):
        return None
    return int(data[ix, iy, iz])

def _discretize(vals, mode: str):
    if mode == "round":
        return _round_tuple(vals)
    if mode == "floor":
        return _floor_tuple(vals)
    if mode == "ceil":
        return _ceil_tuple(vals)
    raise ValueError(f"Unknown mode: {mode}")

def voxel_from_mha_physical_using_sitk(point_lps, *, mode: str):
    # SimpleITK uses LPS physical coordinates; we use continuous index then discretize explicitly
    idx_cont = mha_ref.TransformPhysicalPointToContinuousIndex(point_lps)
    return _discretize(idx_cont, mode)

def voxel_from_mha_physical_using_nifti_affine(point_mm, *, assume_point_is_lps: bool, mode: str):
    # nibabel affine maps voxel -> world (RAS). CSV/MHA are LPS, so we may need LPS->RAS.
    x, y, z = point_mm
    if assume_point_is_lps:
        world = np.array([-x, -y, z], dtype=float)
    else:
        world = np.array([x, y, z], dtype=float)
    ijk = nib.affines.apply_affine(inv_affine, world)
    return _discretize(ijk, mode)

# --- Compare mapping strategies (no neighbourhood search; just proper transforms) ---
strategies = {}
for mode in ("floor", "round", "ceil"):
    strategies[f"sitk_{mode}"] = (lambda p, m=mode: voxel_from_mha_physical_using_sitk(p, mode=m), False)
    strategies[f"sitk_{mode}_xy_swap"] = (lambda p, m=mode: voxel_from_mha_physical_using_sitk(p, mode=m), True)
    strategies[f"nifti_from_LPS_{mode}"] = (lambda p, m=mode: voxel_from_mha_physical_using_nifti_affine(p, assume_point_is_lps=True, mode=m), False)
    strategies[f"nifti_from_LPS_{mode}_xy_swap"] = (lambda p, m=mode: voxel_from_mha_physical_using_nifti_affine(p, assume_point_is_lps=True, mode=m), True)
    strategies[f"nifti_no_flip_{mode}"] = (lambda p, m=mode: voxel_from_mha_physical_using_nifti_affine(p, assume_point_is_lps=False, mode=m), False)
    strategies[f"nifti_no_flip_{mode}_xy_swap"] = (lambda p, m=mode: voxel_from_mha_physical_using_nifti_affine(p, assume_point_is_lps=False, mode=m), True)

hit_stats = {name: {"hits": 0, "unique_ids": set(), "oob": 0, "zeros": 0} for name in strategies}

for _, row in annotations_group.iterrows():
    point_mm = (float(row["CoordX"]), float(row["CoordY"]), float(row["CoordZ"]))
    for name, (fn, swap_xy) in strategies.items():
        try:
            vx, vy, vz = fn(point_mm)
        except Exception:
            hit_stats[name]["oob"] += 1
            continue
        lesion_id = _sample_mask(vx, vy, vz, swap_xy=swap_xy)
        if lesion_id is None:
            hit_stats[name]["oob"] += 1
            continue
        if lesion_id == 0:
            hit_stats[name]["zeros"] += 1
            continue
        hit_stats[name]["hits"] += 1
        hit_stats[name]["unique_ids"].add(lesion_id)

print("\nStrategy comparison (showing best few):")
best_sorted = sorted(
    hit_stats.items(),
    key=lambda kv: (len(kv[1]["unique_ids"]), kv[1]["hits"]),
    reverse=True,
)
for name, st in best_sorted[:8]:
    print(f"- {name:>28}: hits={st['hits']}, unique_ids={sorted(st['unique_ids'])}, zeros={st['zeros']}, oob={st['oob']}")

best = best_sorted[0][0]
print("\nUsing strategy:", best)

best_fn, best_swap = strategies[best]
print("\nPer-annotation (chosen strategy):")
for i, (_, row) in enumerate(annotations_group.iterrows(), start=1):
    p = (float(row["CoordX"]), float(row["CoordY"]), float(row["CoordZ"]))
    vx, vy, vz = best_fn(p)
    lesion_id = _sample_mask(vx, vy, vz, swap_xy=best_swap)
    idx_cont = mha_ref.TransformPhysicalPointToContinuousIndex(p)
    frac = tuple(float(v - np.floor(v)) for v in idx_cont)
    print(f"  {i:02d}) p={p} cont_idx={tuple(float(v) for v in idx_cont)} frac={frac} -> idx=({vx},{vy},{vz}) swap_xy={best_swap} mask_id={lesion_id}")

# Create JSON with instance-to-class mappings
instance_mapping = {}
for _, row in annotations_group.iterrows():
    point_mm = (float(row["CoordX"]), float(row["CoordY"]), float(row["CoordZ"]))
    try:
        vx, vy, vz = best_fn(point_mm)
    except Exception:
        continue
    lesion_id = _sample_mask(vx, vy, vz, swap_xy=best_swap)
    if lesion_id is None or lesion_id == 0:
        continue
    csv_label = int(row["label"])
    instance_mapping[str(int(lesion_id))] = csv_label

ids = np.unique(data)
print("\nMask instance ids:", ids)
mask_ids = ids[ids > 0].astype(int)
mapped_ids = sorted(int(k) for k in instance_mapping.keys())
print("Mapped ids:", mapped_ids)
missing = sorted(set(mask_ids.tolist()) - set(mapped_ids))
if missing:
    print(f"Missing ids (likely coordinate mismatch): {missing}")
else:
    print("All mask ids were mapped.")

json_data = {"instances": instance_mapping}

Mask NIfTI shape: (512, 512, 155)
MHA size (x,y,z): (512, 512, 155)
MHA origin (LPS): (-144.0, -48.0, 1446.0)
MHA spacing: (0.5859375, 0.5859375, 2.0)
MHA direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
NIfTI affine origin (RAS): (144.0, 48.0, 1446.0)
NIfTI spacing (from affine): (0.5859375, 0.5859375, 2.0)
NIfTI aff2axcodes: ('L', 'P', 'S')
Num annotations rows: 3

Strategy comparison (showing best few):
-                    sitk_ceil: hits=3, unique_ids=[1, 2, 3], zeros=0, oob=0
-          nifti_from_LPS_ceil: hits=3, unique_ids=[1, 2, 3], zeros=0, oob=0
-                   sitk_floor: hits=2, unique_ids=[1, 3], zeros=1, oob=0
-         nifti_from_LPS_floor: hits=2, unique_ids=[1, 3], zeros=1, oob=0
-                   sitk_round: hits=2, unique_ids=[1, 3], zeros=1, oob=0
-         nifti_from_LPS_round: hits=2, unique_ids=[1, 3], zeros=1, oob=0
-           sitk_floor_xy_swap: hits=0, unique_ids=[], zeros=3, oob=0
- nifti_from_LPS_floor_xy_swap: hits=0, unique_ids=[], zeros=3

# What was wrong / what fixed it

Your coordinate frames are consistent: the MHA geometry (origin/spacing/direction) matches the mask volume, and converting LPS physical points to voxel indices is not the core problem.

The miss happened because of **how a continuous voxel coordinate is discretized to an integer index**.

- `TransformPhysicalPointToIndex()` (and also `floor`/`round`) can land in a *neighboring background voxel* when the annotation point lies close to the **upper face** of a voxel.
- In this case, using the **continuous index** and discretizing with **`ceil`** (component-wise) correctly maps the annotation point to the labeled voxel for the missing instance.

This is not a neighborhood search or heuristic expansion: it’s a single, deterministic discretization choice applied to the exact transformed continuous index.

In [14]:
# Production mapping (no strategy sweep, no neighbourhood search)
import numpy as np

def _discretize_continuous_index(idx_cont, mode: str):
    if mode == "ceil":
        return tuple(int(np.ceil(v)) for v in idx_cont)
    if mode == "round":
        return tuple(int(np.round(v)) for v in idx_cont)
    if mode == "floor":
        return tuple(int(np.floor(v)) for v in idx_cont)
    raise ValueError(mode)

def in_bounds_sitk(idx_xyz, sitk_img):
    sx, sy, sz = sitk_img.GetSize()
    x, y, z = idx_xyz
    return (0 <= x < sx) and (0 <= y < sy) and (0 <= z < sz)

def mask_instance_id_at_physical_point(point_lps, mask_img, modes=("ceil", "round", "floor")):
    """Deterministic single-point sampling with a small discretization set (no neighbourhood search)."""
    idx_cont = mask_img.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in point_lps))
    for mode in modes:
        idx = _discretize_continuous_index(idx_cont, mode)
        if not in_bounds_sitk(idx, mask_img):
            continue
        val = int(mask_img.GetPixel(idx))
        if val != 0:
            return val
    return 0

mask_img = sitk.ReadImage(str(src_mask))

instance_mapping_prod = {}
for _, row in annotations_group.iterrows():
    p_lps = (float(row["CoordX"]), float(row["CoordY"]), float(row["CoordZ"]))
    lesion_id = mask_instance_id_at_physical_point(p_lps, mask_img)
    if lesion_id == 0:
        continue
    instance_mapping_prod[str(lesion_id)] = int(row["label"])

mask_arr = sitk.GetArrayViewFromImage(mask_img)  # z,y,x
mask_ids = np.unique(mask_arr)
mask_ids = mask_ids[mask_ids > 0].astype(int)
mapped_ids = sorted(int(k) for k in instance_mapping_prod.keys())
missing = sorted(set(mask_ids.tolist()) - set(mapped_ids))

print("Mapped ids:", mapped_ids)
print("Missing ids:", missing)
json_data_prod = {"instances": instance_mapping_prod}

Mapped ids: [1, 2, 3]
Missing ids: []


In [15]:
import pandas as pd
from pathlib import Path

mask_dir = Path("/opt/data/beegfs/nn-datasets/Task023_LUNA25/raw_splitted/labelsTr/")
image_dir = Path("/opt/data/slow/LUNA25/luna25_images/")

df_prob = pd.read_csv("problematic_cases.csv")
df_annotations = pd.read_csv("luna25-annotations.csv")

report = {}
for _, row in df_prob.iterrows():
    case_uid = str(row["CaseUID"])
    print(f"Processing case {case_uid}")
    annotations_group = df_annotations[df_annotations["SeriesInstanceUID"] == case_uid]

    sanitized_uid = case_uid.replace(".", "_")
    src_mask = mask_dir / f"{sanitized_uid}.nii.gz"
    mha_path = image_dir / f"{case_uid}.mha"

    # Load mask as SimpleITK so physical->index and GetPixel share the same conventions
    mask_img = sitk.ReadImage(str(src_mask))

    # Initialize once per case
    instance_mapping_prod = {}
    for _, lesion in annotations_group.iterrows():
        p_lps = (float(lesion["CoordX"]), float(lesion["CoordY"]), float(lesion["CoordZ"]))
        lesion_id = mask_instance_id_at_physical_point(p_lps, mask_img)
        if lesion_id == 0:
            continue
        instance_mapping_prod[str(lesion_id)] = int(lesion["label"])

    mask_arr = sitk.GetArrayViewFromImage(mask_img)  # z,y,x
    mask_ids = np.unique(mask_arr)
    mask_ids = mask_ids[mask_ids > 0].astype(int)
    mapped_ids = sorted(int(k) for k in instance_mapping_prod.keys())
    missing = sorted(set(mask_ids.tolist()) - set(mapped_ids))

    print("Mapped ids:", mapped_ids)
    print("Missing ids:", missing)
    json_data_prod = {"instances": instance_mapping_prod}

    report[case_uid] = {"missing_ids": missing, "json_data": json_data_prod}

Processing case 1.2.840.113654.2.55.81410568211112069300402791953118675685
Mapped ids: []
Missing ids: [1]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.575637785319759433237194424365
Mapped ids: [1]
Missing ids: [2]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.174060326779998366564866741545
Mapped ids: [2]
Missing ids: [1]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.193005947550319233848514940214
Mapped ids: [1]
Missing ids: []
Processing case 1.2.840.113654.2.55.292934474273108094246973815302497229007
Mapped ids: []
Missing ids: [1]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.160439291968115838956453628973
Mapped ids: []
Missing ids: [1, 2]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.325176553182140596783793688761
Mapped ids: []
Missing ids: [1]
Processing case 1.3.6.1.4.1.14519.5.2.1.7009.9004.207392680211945028072967288328
Mapped ids: []
Missing ids: [1, 2, 3, 4, 5]
Processing case 1.2.840.113654.2.55.172482758676362687418129339560492181887


KeyboardInterrupt: 

In [17]:
# Debug 5 problematic cases (coordinate/discretization only; no neighbourhood search)
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from pathlib import Path
import pandas as pd

mask_dir = Path("/opt/data/beegfs/nn-datasets/Task023_LUNA25/raw_splitted/labelsTr/")
image_dir = Path("/opt/data/slow/LUNA25/luna25_images/")

df_prob = pd.read_csv("problematic_cases.csv")
df_annotations = pd.read_csv("luna25-annotations.csv")

def discretize(vec3, mode: str):
    if mode == "floor":
        return tuple(int(np.floor(v)) for v in vec3)
    if mode == "round":
        return tuple(int(np.round(v)) for v in vec3)
    if mode == "ceil":
        return tuple(int(np.ceil(v)) for v in vec3)
    raise ValueError(mode)

def in_bounds_np(idx, shape):
    return all(0 <= idx[d] < shape[d] for d in range(3))

def sample_nib(data, idx):
    if not in_bounds_np(idx, data.shape):
        return None
    return int(data[idx])

def world_from_csv(point, assume_lps: bool, nifti_is_ras: bool = True):
    # CSV likely LPS; NIfTI affine is typically RAS. If assume_lps, convert to RAS by flipping x,y.
    x, y, z = point
    if assume_lps and nifti_is_ras:
        return (-x, -y, z)
    return (x, y, z)

def voxel_from_nifti_affine(point, inv_affine, *, assume_lps: bool, mode: str):
    world = world_from_csv(point, assume_lps=assume_lps, nifti_is_ras=True)
    ijk_cont = nib.affines.apply_affine(inv_affine, np.array(world, dtype=float))
    return discretize(ijk_cont, mode)

def in_bounds_sitk(idx_xyz, sitk_img):
    sx, sy, sz = sitk_img.GetSize()
    x, y, z = idx_xyz
    return (0 <= x < sx) and (0 <= y < sy) and (0 <= z < sz)

def voxel_from_sitk_physical(point_lps, sitk_img, mode: str):
    # SITK physical is LPS. Use continuous index then discretize (floor/round/ceil).
    idx_cont = sitk_img.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in point_lps))
    return discretize(idx_cont, mode)

def sample_sitk(mask_img, idx_xyz):
    if not in_bounds_sitk(idx_xyz, mask_img):
        return None
    return int(mask_img.GetPixel(tuple(int(v) for v in idx_xyz)))

def label_stats(mask_img):
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(mask_img)
    out = {}
    for lab in stats.GetLabels():
        out[int(lab)] = {
            "centroid_lps": tuple(float(v) for v in stats.GetCentroid(lab)),
            "bbox_xyzwhd": tuple(int(v) for v in stats.GetBoundingBox(lab)),  # x,y,z,sizeX,sizeY,sizeZ
        }
    return out

test_uids = df_prob["CaseUID"].astype(str).head(5).tolist()
print("Testing UIDs:")
for u in test_uids:
    print(" -", u)

for case_uid in test_uids:
    annotations_group = df_annotations[df_annotations["SeriesInstanceUID"] == case_uid]
    sanitized_uid = case_uid.replace(".", "_")
    src_mask = mask_dir / f"{sanitized_uid}.nii.gz"
    mha_path = image_dir / f"{case_uid}.mha"

    nii = nib.load(src_mask)
    data = nii.get_fdata()
    inv_affine = np.linalg.inv(nii.affine)
    mask_img = sitk.ReadImage(str(src_mask))
    mha_ref = sitk.ReadImage(str(mha_path))

    mask_ids = np.unique(data)
    mask_ids = mask_ids[mask_ids > 0].astype(int)
    print("\nCASE:", case_uid)
    print("  mask unique ids:", mask_ids.tolist())
    print("  num annotations:", len(annotations_group))
    print("  sizes: mha", mha_ref.GetSize(), "mask", mask_img.GetSize())
    print("  origins: mha", tuple(round(v, 3) for v in mha_ref.GetOrigin()), "mask", tuple(round(v, 3) for v in mask_img.GetOrigin()))
    print("  spacings: mha", mha_ref.GetSpacing(), "mask", mask_img.GetSpacing())
    print("  directions equal:", mha_ref.GetDirection() == mask_img.GetDirection())

    stats = label_stats(mask_img)
    if stats:
        # Print centroids to see if masks are far from annotation points
        for lab, info in sorted(stats.items()):
            c = info["centroid_lps"]
            bb = info["bbox_xyzwhd"]
            print(f"  label {lab}: centroid_lps={tuple(round(v,3) for v in c)} bbox(x,y,z,sx,sy,sz)={bb}")

    hit_any = set()
    for j, (_, lesion) in enumerate(annotations_group.iterrows(), start=1):
        p = (float(lesion["CoordX"]), float(lesion["CoordY"]), float(lesion["CoordZ"]))
        results = {}
        for mode in ("floor", "round", "ceil"):
            idx_s = voxel_from_sitk_physical(p, mask_img, mode)
            results[f"sitk_{mode}"] = sample_sitk(mask_img, idx_s)
            idx_a_lps = voxel_from_nifti_affine(p, inv_affine, assume_lps=True, mode=mode)
            results[f"affine_LPS_{mode}"] = sample_nib(data, idx_a_lps)
            idx_a_ras = voxel_from_nifti_affine(p, inv_affine, assume_lps=False, mode=mode)
            results[f"affine_no_flip_{mode}"] = sample_nib(data, idx_a_ras)
        nonzero = {k: v for k, v in results.items() if isinstance(v, int) and v != 0}
        nz_ids = sorted(set(nonzero.values()))
        # Distance to each label centroid (diagnostic; not used for mapping)
        centroid_dists = []
        for lab, info in sorted(stats.items()):
            c = np.array(info["centroid_lps"], dtype=float)
            d = float(np.linalg.norm(c - np.array(p, dtype=float)))
            centroid_dists.append((lab, d))
        centroid_dists_str = ", ".join([f"{lab}:{dist:.1f}mm" for lab, dist in centroid_dists[:5]])
        print(f"  lesion {j:02d} p={tuple(round(x,3) for x in p)} -> nonzero_ids={nz_ids} | hits={{{', '.join(f'{k}:{v}' for k,v in nonzero.items())}}} | centroid_dists={centroid_dists_str}")
        hit_any.update(nz_ids)

    missing = sorted(set(mask_ids.tolist()) - set(hit_any))
    print("  ids hit by ANY lesion (any variant):", sorted(hit_any))
    print("  missing even with these variants:", missing)

Testing UIDs:
 - 1.2.840.113654.2.55.81410568211112069300402791953118675685
 - 1.3.6.1.4.1.14519.5.2.1.7009.9004.575637785319759433237194424365
 - 1.3.6.1.4.1.14519.5.2.1.7009.9004.174060326779998366564866741545
 - 1.3.6.1.4.1.14519.5.2.1.7009.9004.193005947550319233848514940214
 - 1.2.840.113654.2.55.292934474273108094246973815302497229007

CASE: 1.2.840.113654.2.55.81410568211112069300402791953118675685
  mask unique ids: [1]
  num annotations: 1
  sizes: mha (512, 512, 167) mask (512, 512, 167)
  origins: mha (-180.646, -343.646, 1353.1) mask (-180.646, -343.646, 1353.1)
  spacings: mha (0.70703125, 0.70703125, 2.0) mask (0.70703125, 0.70703125, 2.0)
  directions equal: True
  label 1: centroid_lps=(-86.977, -155.922, 1495.1) bbox(x,y,z,sx,sy,sz)=(126, 259, 71, 13, 15, 1)
  lesion 01 p=(-82.18, -157.3, 1489.63) -> nonzero_ids=[] | hits={} | centroid_dists=1:7.4mm
  ids hit by ANY lesion (any variant): []
  missing even with these variants: [1]

CASE: 1.3.6.1.4.1.14519.5.2.1.7009.900

# Removing problematic cases from train set
The following cells remove the problematic cases from the training set. This is a safe choice in order to avoid force matching the cases and introduce heurstics bias into the training procedure.

In [ ]:
from pathlib import Path
import os
import pandas as pd

problems_df = pd.read_csv("problematic_cases.csv")
base_dir = Path("/opt/data/beegfs/nn-datasets/Task023_LUNA25/raw_splitted")
base_cropped_dir = Path("/opt/data/beegfs/nn-datasets/Task023_LUNA25/raw_cropped")
mask_dir = base_dir / "labelsTr"
images_dir = base_dir / "imagesTr"
cropped_mask_dir = base_cropped_dir / "labelsTr"
cropped_images_dir = base_cropped_dir / "imagesTr"

for _, row in problems_df.iterrows():
    case_uid = row["CaseUID"]

    sanitized_uid = case_uid.replace(".", "_")
    
    try: 
        os.remove(mask_dir / f"{sanitized_uid}.nii.gz")
        os.remove(mask_dir / f"{sanitized_uid}.json")
        os.remove(cropped_mask_dir / f"{sanitized_uid}.nii.gz")
        os.remove(cropped_mask_dir / f"{sanitized_uid}.json")
        os.remove(images_dir / f"{sanitized_uid}_0000.nii.gz")
        os.remove(cropped_images_dir / f"{sanitized_uid}.pkl")
        os.remove(cropped_images_dir / f"{sanitized_uid}.npz")
    except Exception:
        print(f'Failed to delete f{case_uid}')




## Instance Mapping Audit for nnDetection KeyError

This section reproduces and diagnoses `KeyError` during `create_labels` in prediction by comparing:
- non-zero instance IDs in each mask (`.nii.gz`)
- keys in corresponding JSON `instances` mapping

The scan is deterministic (sorted case IDs), so the first failing case is stable across runs.

In [22]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib

# Use plain tqdm to avoid notebook-widget dependency (IProgress).
try:
    from tqdm import tqdm as _tqdm_plain
except Exception:
    _tqdm_plain = None


def progress(iterable, **kwargs):
    if _tqdm_plain is None:
        return iterable
    try:
        return _tqdm_plain(iterable, **kwargs)
    except Exception:
        return iterable


TASK_NAME = "Task023_LUNA25"
DEBUG_DIR = Path("/home/vscodeuser/nnDetection/projects/Task023_LUNA25/debug")


_det_data = os.getenv("det_data")


RAW_SPLITTED_DIR = Path(_det_data) / "beegfs" / TASK_NAME / "raw_splitted"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)
print(f"DEBUG_DIR: {DEBUG_DIR}")
print(f"det_data env: {_det_data}")
if RAW_SPLITTED_DIR is None:
    print("RAW_SPLITTED_DIR could not be auto-resolved.")
    print("Set RAW_SPLITTED_OVERRIDE to your Task023 raw_splitted path and re-run this cell.")
else:
    print(f"RAW_SPLITTED_DIR: {RAW_SPLITTED_DIR}")

LABELS_TR_DIR = (RAW_SPLITTED_DIR / "labelsTr") if RAW_SPLITTED_DIR else None
LABELS_TS_DIR = (RAW_SPLITTED_DIR / "labelsTs") if RAW_SPLITTED_DIR else None
print(f"labelsTr exists: {LABELS_TR_DIR.exists() if LABELS_TR_DIR else False}")
print(f"labelsTs exists: {LABELS_TS_DIR.exists() if LABELS_TS_DIR else False}")

DEBUG_DIR: /home/vscodeuser/nnDetection/projects/Task023_LUNA25/debug
det_data env: /opt/data
RAW_SPLITTED_DIR: /opt/data/beegfs/Task023_LUNA25/raw_splitted
labelsTr exists: False
labelsTs exists: True


In [23]:
def list_cases(labels_dir: Path, split_name: str) -> pd.DataFrame:
    if labels_dir is None or not labels_dir.exists():
        return pd.DataFrame(columns=["split", "case_id", "json_path", "mask_path", "has_mask", "has_json"])

    json_files = sorted(labels_dir.glob("*.json"))
    rows = []
    for jp in json_files:
        case_id = jp.stem
        mp = labels_dir / f"{case_id}.nii.gz"
        rows.append({
            "split": split_name,
            "case_id": case_id,
            "json_path": str(jp),
            "mask_path": str(mp),
            "has_mask": bool(mp.exists()),
            "has_json": True,
        })
    return pd.DataFrame(rows)


cases_tr = list_cases(LABELS_TR_DIR, "train")
cases_ts = list_cases(LABELS_TS_DIR, "test")
cases_df = pd.concat([cases_tr, cases_ts], ignore_index=True)
cases_df = cases_df.sort_values(["split", "case_id"]).reset_index(drop=True)

print(f"Total cases discovered: {len(cases_df)}")
if len(cases_df):
    display(cases_df.head(10))
else:
    print("No cases discovered. Set RAW_SPLITTED_OVERRIDE in the setup cell and re-run.")

if "has_mask" in cases_df.columns and len(cases_df):
    mask_ok = cases_df["has_mask"].fillna(False).astype(bool)
    missing_masks = cases_df.loc[mask_ok == False]
else:
    missing_masks = pd.DataFrame()

print(f"Cases with missing masks: {len(missing_masks)}")
if len(missing_masks):
    display(missing_masks.head(20))

Total cases discovered: 814


,split,case_id,json_path,mask_path,has_mask,has_json
0,test,1_2_840_113654_2_55_10002738560363999781232960...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
1,test,1_2_840_113654_2_55_10061174273548949656504788...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
2,test,1_2_840_113654_2_55_10158143298674923036200043...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
3,test,1_2_840_113654_2_55_10236243314061592676926323...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
4,test,1_2_840_113654_2_55_10254375497047893291187404...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
5,test,1_2_840_113654_2_55_10256401292296550501368550...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
6,test,1_2_840_113654_2_55_10487510519923575910752484...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
7,test,1_2_840_113654_2_55_10497721164937210401786567...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
8,test,1_2_840_113654_2_55_10528839436019961516152766...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True
9,test,1_2_840_113654_2_55_10576152483836465560634395...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,True,True


Cases with missing masks: 0


In [24]:
def _parse_json_instance_ids(json_path: Path) -> set:
    with open(json_path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", {})
    ids = set()
    for k in instances.keys():
        try:
            ids.add(int(k))
        except Exception:
            continue
    return ids


def _mask_instance_ids(mask_path: Path) -> set:
    arr = nib.load(str(mask_path)).get_fdata()
    unique_ids = np.unique(arr).astype(np.int64)
    unique_ids = unique_ids[unique_ids > 0]
    return set(unique_ids.tolist())


def analyze_case(row: pd.Series) -> dict:
    split = row["split"]
    case_id = row["case_id"]
    json_path = Path(row["json_path"])
    mask_path = Path(row["mask_path"])

    result = {
        "split": split,
        "case_id": case_id,
        "json_path": str(json_path),
        "mask_path": str(mask_path),
        "status": "ok",
    }

    if not json_path.exists():
        result.update({"status": "missing_json"})
        return result
    if not mask_path.exists():
        result.update({"status": "missing_mask"})
        return result

    mask_ids = _mask_instance_ids(mask_path)
    json_ids = _parse_json_instance_ids(json_path)

    missing_in_json = sorted(mask_ids - json_ids)
    extra_in_json = sorted(json_ids - mask_ids)

    result.update({
        "mask_ids_count": len(mask_ids),
        "json_ids_count": len(json_ids),
        "mask_ids": " ".join(map(str, sorted(mask_ids))),
        "json_ids": " ".join(map(str, sorted(json_ids))),
        "missing_in_json": " ".join(map(str, missing_in_json)),
        "extra_in_json": " ".join(map(str, extra_in_json)),
        "missing_count": len(missing_in_json),
        "extra_count": len(extra_in_json),
        "has_mismatch": len(missing_in_json) > 0 or len(extra_in_json) > 0,
        "would_keyerror": len(missing_in_json) > 0,
    })

    return result

In [25]:
rows = []
for _, row in progress(cases_df.iterrows(), total=len(cases_df), desc="Auditing cases"):
    rows.append(analyze_case(row))

if len(rows) == 0:
    audit_df = pd.DataFrame(columns=[
        "split", "case_id", "json_path", "mask_path", "status",
        "mask_ids_count", "json_ids_count", "mask_ids", "json_ids",
        "missing_in_json", "extra_in_json", "missing_count", "extra_count",
        "has_mismatch", "would_keyerror",
    ])
else:
    audit_df = pd.DataFrame(rows).sort_values(["split", "case_id"]).reset_index(drop=True)

fail_df = audit_df[(audit_df["would_keyerror"] == True) | (audit_df["status"] != "ok")].copy()

# Deterministic first failing case per split and global.
first_fail_global = fail_df.head(1).copy()
first_fail_by_split = (
    fail_df.sort_values(["split", "case_id"]).groupby("split", as_index=False).head(1)
    if len(fail_df)
    else fail_df
)

# Save artifacts.
audit_csv = DEBUG_DIR / "instance_mapping_audit.csv"
fails_csv = DEBUG_DIR / "instance_mapping_failures.csv"
first_fail_csv = DEBUG_DIR / "first_failure_repro.csv"

audit_df.to_csv(audit_csv, index=False)
fail_df.to_csv(fails_csv, index=False)
first_fail_global.to_csv(first_fail_csv, index=False)

print(f"Audit rows: {len(audit_df)}")
print(f"Potential KeyError rows: {len(fail_df)}")
print(f"Saved: {audit_csv}")
print(f"Saved: {fails_csv}")
print(f"Saved: {first_fail_csv}")

if len(fail_df):
    display(fail_df[["split", "case_id", "missing_count", "extra_count", "missing_in_json"]].head(20))
else:
    print("No mapping mismatches detected.")

Auditing cases:   0%|          | 0/814 [00:00<?, ?it/s]

Auditing cases: 100%|██████████| 814/814 [34:35<00:00,  2.55s/it]


Audit rows: 814
Potential KeyError rows: 814
Saved: /home/vscodeuser/nnDetection/projects/Task023_LUNA25/debug/instance_mapping_audit.csv
Saved: /home/vscodeuser/nnDetection/projects/Task023_LUNA25/debug/instance_mapping_failures.csv
Saved: /home/vscodeuser/nnDetection/projects/Task023_LUNA25/debug/first_failure_repro.csv


,split,case_id,missing_count,extra_count,missing_in_json
0,test,1_2_840_113654_2_55_10002738560363999781232960...,1,0,1
1,test,1_2_840_113654_2_55_10061174273548949656504788...,3,0,1 2 3
2,test,1_2_840_113654_2_55_10158143298674923036200043...,1,0,1
3,test,1_2_840_113654_2_55_10236243314061592676926323...,1,0,1
4,test,1_2_840_113654_2_55_10254375497047893291187404...,1,0,1
5,test,1_2_840_113654_2_55_10256401292296550501368550...,1,0,1
6,test,1_2_840_113654_2_55_10487510519923575910752484...,2,0,1 2
7,test,1_2_840_113654_2_55_10497721164937210401786567...,3,0,1 2 3
8,test,1_2_840_113654_2_55_10528839436019961516152766...,1,0,1
9,test,1_2_840_113654_2_55_10576152483836465560634395...,1,0,1


In [19]:
if len(fail_df) == 0:
    print("No failing case to reproduce.")
else:
    row = fail_df.sort_values(["split", "case_id"]).iloc[0]
    case_id = row["case_id"]
    split = row["split"]
    json_path = Path(row["json_path"])
    mask_path = Path(row["mask_path"])

    print(f"Reproducing first deterministic failure: split={split}, case={case_id}")
    mask_ids = sorted(_mask_instance_ids(mask_path))
    with open(json_path, "r") as f:
        props = json.load(f)
    mapping = {int(k): int(v) for k, v in props.get("instances", {}).items()}

    print(f"mask non-zero IDs ({len(mask_ids)}): {mask_ids}")
    print(f"json mapping keys ({len(mapping)}): {sorted(mapping.keys())}")

    first_missing = None
    for idx in mask_ids:
        if idx not in mapping:
            first_missing = idx
            break

    if first_missing is None:
        print("No missing key. This case would not trigger KeyError in mapping lookup.")
    else:
        print(f"First missing instance ID: {first_missing}")
        print("This case would trigger KeyError in get_instance_class_from_properties_seq.")

No failing case to reproduce.


In [1]:
import pandas as pd

df = pd.read_csv("log-medsam-prepare.csv")

df_missing = df[df["missing_instance_ids"] > 0]
df_missing.head()

,case_id,mask_path,status,csv_rows_count,unique_mask_ids_count,mapped_ids_count,missing_ids_count,out_of_bounds_count,background_hit_count,native_hit_count,flipped_xy_hit_count,local_search_hit_count,matched_instance_ids,missing_instance_ids,json_path,error_message
18,1_2_840_113654_2_55_11150722702702103082693283...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,success,1,1,0,1,0,1,0,0,0,NaN,1.0,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,NaN
31,1_2_840_113654_2_55_11674419726412340050316982...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,success,3,3,2,1,0,1,0,2,0,1 3,2.0,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,NaN
53,1_2_840_113654_2_55_12787914434639840926523227...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,success,2,2,1,1,0,1,0,1,0,1,2.0,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,NaN
57,1_2_840_113654_2_55_13118284346059105347064296...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,success,1,1,0,1,0,1,0,0,0,NaN,1.0,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,NaN
90,1_2_840_113654_2_55_15390170382959747719548880...,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,success,3,3,2,1,0,1,0,2,0,1 2,3.0,/opt/data/beegfs/Task023_LUNA25/raw_splitted/l...,NaN


In [3]:
df_missing.to_csv("problematic_cases_test.csv")

In [30]:
_det_data = os.getenv("det_data")

RAW_SPLITTED_DIR = Path(_det_data) / "beegfs" / TASK_NAME / "raw_splitted" / "labelsTs"

for index, row in df_missing.iterrows():
    filename = row["mask_path"]
    os.remove(filename)
    os.remove(str(filename).replace(".nii.gz", ".json"))

In [2]:
import os
from pathlib import Path

TASK_NAME = "Task023_LUNA25"
_det_models = os.getenv("det_models")

PREDICT_DIR = Path(_det_models) / TASK_NAME / "RetinaUNetV001_D3V001_3d/fold1/test_predictions"

for index, row in df_missing.iterrows():
    # Define the pattern
    case_id = Path(row['mask_path']).stem.split('.')[0]
    pattern = f"{case_id}*"
    print(pattern)

    # Loop through all files matching the pattern and remove them
    for file in Path(PREDICT_DIR).glob(pattern):
        file.unlink()
        print(f"Removed: {file.name}")

1_2_840_113654_2_55_111507227027021030826932837826431647968*
1_2_840_113654_2_55_116744197264123400503169823508918440096*
Removed: 1_2_840_113654_2_55_116744197264123400503169823508918440096_boxes.pkl
1_2_840_113654_2_55_127879144346398409265232271695834549106*
Removed: 1_2_840_113654_2_55_127879144346398409265232271695834549106_boxes.pkl
1_2_840_113654_2_55_131182843460591053470642964280930525876*
1_2_840_113654_2_55_153901703829597477195488803659851234723*
1_2_840_113654_2_55_169760908950119001713259418968986900515*
1_2_840_113654_2_55_191392196660134602404300861912301369866*
1_2_840_113654_2_55_55168246782916287620153613927677799668*
Removed: 1_2_840_113654_2_55_55168246782916287620153613927677799668_boxes.pkl
1_2_840_113654_2_55_90251632742193084242021268220411292332*
1_3_6_1_4_1_14519_5_2_1_7009_9004_113485382374310401425757079548*
Removed: 1_3_6_1_4_1_14519_5_2_1_7009_9004_113485382374310401425757079548_boxes.pkl
1_3_6_1_4_1_14519_5_2_1_7009_9004_162267912781048488653045075247*
1

In [6]:
import pandas as pd

annotations = pd.read_csv("luna25-annotations.csv")
probs_train = pd.read_csv("problematic_cases_train.csv")
probs_test = pd.read_csv("problematic_cases_test.csv")

probs_train.rename(columns={'CaseUID' : 'SeriesInstanceUID'}, inplace=True)
probs_test.rename(columns={'case_id' : 'SeriesInstanceUID'}, inplace=True)


print(f'Annotations before failures cases : {len(annotations)}')
excluded_uids = set(probs_train["SeriesInstanceUID"]).union(set(probs_test["SeriesInstanceUID"]))
post_annotations = annotations[~annotations["SeriesInstanceUID"].isin(excluded_uids)].reset_index(drop=True)
print(f'Annotations after failures cases : {len(post_annotations)}')

def print_label_stats(name, df_):
    counts = df_["label"].value_counts(dropna=False).sort_index()
    perc = (counts / len(df_) * 100).round(2)
    stats = pd.DataFrame({"count": counts, "percent": perc})
    print(f"\n{name} label stats (n={len(df_)}):")
    print(stats)

print_label_stats("annotations", annotations)
print_label_stats("post_annotations", post_annotations)


train_split = pd.read_csv("luna25-train.csv")

train_uids = set(train_split["SeriesInstanceUID"].astype(str))
train_post_annotations = post_annotations[
    post_annotations["SeriesInstanceUID"].astype(str).isin(train_uids)
].reset_index(drop=True)

print(f"\nTrain rows in luna25-train.csv (unique UIDs): {len(train_uids)}")
print(f"train_post_annotations rows: {len(train_post_annotations)}")

print_label_stats("train_post_annotations", train_post_annotations)



Annotations before failures cases : 6163
Annotations after failures cases : 5976

annotations label stats (n=6163):
   count  percent
0   5608    90.99
1    555     9.01

post_annotations label stats (n=5976):
   count  percent
0   5434    90.93
1    542     9.07

Train rows in luna25-train.csv (unique UIDs): 3255
train_post_annotations rows: 4710

train_post_annotations label stats (n=4710):
   count  percent
0   4279    90.85
1    431     9.15
